In [ ]:
import numpy as np

In [ ]:
def normalise(x):
    return (x - np.mean(x, axis=0, keepdims=True)) / np.std(x, axis=0, keepdims=True)

In [ ]:
# band pass filter
from scipy.signal import butter, filtfilt, hilbert


def band_pass_filter(data, lowcut, highcut, order=3):
    nyquist = 0.5 * data.shape[1]
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, data, axis=1)

In [ ]:
# generate some fake data featuring oscillations
sampling_freq = 256
time = 180
C = 5

N = time * sampling_freq

x = np.cumsum(np.random.normal(size=(C, N)), axis=1) + 3 * np.sin(np.linspace(0, N/10 * np.pi, N)) + np.random.normal(size=(C, N))
x = band_pass_filter(x, lowcut=20, highcut=500)

In [ ]:
# hilbert transform and take the modulus
x_hilbert_modulus = np.abs(hilbert(x))

In [ ]:
def get_rms_resids(x, window_len=50, window_overlap=0.5):
    # Implement the DFA (Detrended Fluctuation Analysis) algorithm here
    y = np.cumsum(x - np.mean(x))  # Integrate the signal after removing the mean
    n = len(y)

    step = int(window_len * (1 - window_overlap))
    F = []  # Initialize an empty list to store RMS residuals

    for i in range(0, n, step):
        if i + window_len > n:
            break
        segment = y[i:i+window_len]
        t = np.arange(window_len)
        p = np.polyfit(t, segment, 1)  # Linear detrending
        trend = p[0] * t + p[1]
        F.append(np.sqrt(np.mean((segment - trend)**2)))

    return np.array(F)

def dfa(x, min_window=10, max_window=100, window_steps=10, window_overlap=0.5):
    windows = np.arange(min_window, max_window + 1, window_steps)
    Fs = []
    ns = []
    for window_len in windows:
        F = get_rms_resids(x, window_len, window_overlap)
        Fs.append(F)
        ns.append(np.repeat(window_len, len(F)))

    Fs = np.concatenate(Fs)
    ns = np.concatenate(ns)

    # Convert to log-log scale for DFA analysis
    Fs = np.log(Fs)
    ns = np.log(ns)

    # Perform linear regression on the log-log data to estimate the DFA exponent
    p = np.polyfit(ns, Fs, 1)  # Linear regression on log-log data

    alpha = p[0]  # DFA exponent
    print(f'DFA exponent (alpha): {alpha}')

    return alpha

def fei(x, window_len=100, window_overlap=0.5):
    y = np.cumsum(x - np.mean(x))

    step = int(window_len * (1 - window_overlap))
    mean_amps = []
    rms_resids = []

    for i in range(0, len(y) - window_len + 1, step):
        segment = y[i:i+window_len]
        mean_amp = np.mean(segment)
        mean_amps.append(mean_amp)

        norm_seg = segment / mean_amp
        t = np.arange(window_len)
        p = np.polyfit(t, norm_seg, 1)  # Linear detrending
        trend = p[0] * t + p[1]
        norm_seg_detrended = norm_seg - trend

        rms_resid = np.sqrt(np.mean(norm_seg_detrended**2))
        rms_resids.append(rms_resid)

    fei_value = 1 - np.corrcoef(mean_amps, rms_resids)[0, 1]

    print(f'FEI value: {fei_value}')

    return fei_value




In [ ]:
ix = 3

In [ ]:
fei(x_hilbert_modulus[ix, :], window_len=sampling_freq * 8, window_overlap=0.5);

In [ ]:
dfa(x_hilbert_modulus[ix, :], window_overlap=0.5, min_window=1000, max_window=5000, window_steps=200);

In [ ]:
from crosci.biomarkers import (
    DFA,
    fEI,
)

In [ ]:
(DFA_array, window_sizes, fluctuations, DFA_intercept) = DFA(
    x_hilbert_modulus[ix, :][None, :], 
    sampling_freq,
    [4, 20],
    [4, 20],
    overlap=True,
)

DFA_array[0]

In [ ]:
(fEI_outliers_removed, fEI_val, num_outliers, wAmp, wDNF) = fEI(
    x_hilbert_modulus[ix, :][None, :],
    sampling_freq,
    8,
    0.5,
    DFA_array
)

fEI_val